In [9]:
import os
import glob
import json
import open3d as o3d
import numpy as np
import cv2

# === CHANGE THIS: folder with rgb_*.png, depth_*.png, masks/ ===
frames_dir = r"C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001"
mask_dir   = os.path.join(frames_dir, "masks")

# --- Load intrinsics from intrinsics.json (same as before) ---
meta_path = os.path.join(frames_dir, "intrinsics.json")
with open(meta_path, "r") as f:
    meta = json.load(f)

fx = float(meta["fx"])
fy = float(meta["fy"])
cx = float(meta["cx"])
cy = float(meta["cy"])
W  = int(meta["width"])
H  = int(meta["height"])

# depth_scale in meters per unit (e.g. 0.001 if depth is in mm)
depth_scale_m_per_unit = float(meta["depth_scale"])

print("Intrinsics:")
print(f"fx={fx}, fy={fy}, cx={cx}, cy={cy}, W={W}, H={H}, depth_scale={depth_scale_m_per_unit}")

# --- Get file lists ---
rgb_files   = sorted(glob.glob(os.path.join(frames_dir, "rgb_*.png")))
depth_files = sorted(glob.glob(os.path.join(frames_dir, "depth_*.png")))

print("Found RGB frames:", len(rgb_files))
print("Found depth frames:", len(depth_files))

if not rgb_files or not depth_files:
    raise RuntimeError("No rgb_*.png or depth_*.png files found. Check frames_dir.")

def mask_for(rgb_path: str) -> str:
    name = os.path.basename(rgb_path)  # e.g. rgb_0001.png
    return os.path.join(mask_dir, name.replace("rgb_", "mask_"))

# Pick one test frame
test_rgb   = rgb_files[0]
test_depth = depth_files[0]
test_mask  = mask_for(test_rgb)

print("Test RGB:  ", test_rgb)
print("Test depth:", test_depth)
print("Test mask:", test_mask, "(exists:", os.path.exists(test_mask), ")")


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
Intrinsics:
fx=646.2672119140625, fy=645.5238037109375, cx=641.302490234375, cy=360.6357116699219, W=1280, H=720, depth_scale=0.0010000000474974513
Found RGB frames: 43
Found depth frames: 43
Test RGB:   C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\rgb_0001.png
Test depth: C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\depth_0001.png
Test mask: C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\masks\mask_0001.png (exists: True )


In [10]:
def write_ply_xyz(path, points):
    """
    Save Nx3 numpy array as ASCII PLY (x y z only).
    points: shape (N, 3)
    """
    points = points.astype(np.float32)
    n = points.shape[0]
    header = [
        "ply",
        "format ascii 1.0",
        f"element vertex {n}",
        "property float x",
        "property float y",
        "property float z",
        "end_header",
    ]
    with open(path, "w") as f:
        f.write("\n".join(header) + "\n")
        for x, y, z in points:
            f.write(f"{x} {y} {z}\n")
    print("PLY written with", n, "points ->", path)


In [11]:
# --- Load images ---
color = cv2.imread(test_rgb, cv2.IMREAD_COLOR)
depth = cv2.imread(test_depth, cv2.IMREAD_UNCHANGED)

if color is None:
    raise RuntimeError(f"Failed to read RGB image: {test_rgb}")
if depth is None:
    raise RuntimeError(f"Failed to read depth image: {test_depth}")

if color.shape[1] != W or color.shape[0] != H:
    raise RuntimeError(f"RGB size {color.shape[1]}x{color.shape[0]} != intrinsics {W}x{H}")
if depth.shape[1] != W or depth.shape[0] != H:
    raise RuntimeError(f"Depth size {depth.shape[1]}x{depth.shape[0]} != intrinsics {W}x{H}")

# Mask (optional but recommended)
if os.path.exists(test_mask):
    mask = cv2.imread(test_mask, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        raise RuntimeError(f"Failed to read mask: {test_mask}")
    if mask.shape[1] != W or mask.shape[0] != H:
        raise RuntimeError(f"Mask size {mask.shape[1]}x{mask.shape[0]} != intrinsics {W}x{H}")
else:
    # If no mask, treat everything as valid
    mask = np.ones((H, W), dtype=np.uint8) * 255

print("RGB shape:", color.shape, "Depth shape:", depth.shape, "Mask shape:", mask.shape)

# Ensure depth is float32 for math
depth = depth.astype(np.float32)

# Valid pixels: mask > 0 AND depth > 0
valid = (mask > 0) & (depth > 0)

v_idx, u_idx = np.where(valid)  # v = row (y), u = col (x)
print("Valid pixels before subsampling:", len(v_idx))

# Subsample to avoid huge memory usage
step = 4  # take every 4th valid point
v_idx = v_idx[::step]
u_idx = u_idx[::step]
print("Valid pixels after subsampling:", len(v_idx))

# Corresponding depth values (in meters)
Z = depth[v_idx, u_idx] * depth_scale_m_per_unit  # shape (N,)
# Avoid zeros (just in case)
nonzero = Z > 0
Z = Z[nonzero]
v_idx = v_idx[nonzero]
u_idx = u_idx[nonzero]
print("Non-zero depth points:", len(Z))

# Camera coordinates:
# X = (u - cx) * Z / fx
# Y = (v - cy) * Z / fy
X = (u_idx - cx) * Z / fx
Y = (v_idx - cy) * Z / fy

points = np.stack([X, Y, Z], axis=1)  # (N, 3)
print("Point cloud shape:", points.shape)

# Save as PLY
out_pcd_path = os.path.join(frames_dir, "ffb_single_frame_numpy.ply")
write_ply_xyz(out_pcd_path, points)


RGB shape: (720, 1280, 3) Depth shape: (720, 1280) Mask shape: (720, 1280)
Valid pixels before subsampling: 29871
Valid pixels after subsampling: 7468
Non-zero depth points: 7468
Point cloud shape: (7468, 3)
PLY written with 7468 points -> C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\ffb_single_frame_numpy.ply


In [12]:
all_points = []

# Use every Nth frame to keep it light
frame_step = 5

for idx in range(0, min(len(rgb_files), len(depth_files)), frame_step):
    rgb_path   = rgb_files[idx]
    depth_path = depth_files[idx]
    mask_path  = mask_for(rgb_path)

    print(f"[{idx}/{len(rgb_files)}] Processing", os.path.basename(rgb_path))

    color = cv2.imread(rgb_path, cv2.IMREAD_COLOR)
    depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
    if color is None or depth is None:
        print("  Skipping (failed to load).")
        continue

    if os.path.exists(mask_path):
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    else:
        mask = np.ones((H, W), dtype=np.uint8) * 255  # no mask -> all valid

    depth = depth.astype(np.float32)
    valid = (mask > 0) & (depth > 0)

    v_idx, u_idx = np.where(valid)

    # Subsample aggressively per frame
    step = 6
    v_idx = v_idx[::step]
    u_idx = u_idx[::step]

    Z = depth[v_idx, u_idx] * depth_scale_m_per_unit
    nonzero = Z > 0
    Z = Z[nonzero]
    v_idx = v_idx[nonzero]
    u_idx = u_idx[nonzero]

    X = (u_idx - cx) * Z / fx
    Y = (v_idx - cy) * Z / fy

    pts = np.stack([X, Y, Z], axis=1)
    all_points.append(pts)

if not all_points:
    raise RuntimeError("No points collected from any frame!")

all_points = np.concatenate(all_points, axis=0)
print("Total merged points before final subsampling:", all_points.shape)

# Final subsample if still huge
if all_points.shape[0] > 200_000:
    all_points = all_points[::int(all_points.shape[0] / 200_000) + 1]

print("Total merged points after subsampling:", all_points.shape)

out_merged_path = os.path.join(frames_dir, "ffb_multi_frame_numpy.ply")
write_ply_xyz(out_merged_path, all_points)


[0/43] Processing rgb_0001.png
[5/43] Processing rgb_0006.png
[10/43] Processing rgb_0011.png
[15/43] Processing rgb_0016.png
[20/43] Processing rgb_0021.png
[25/43] Processing rgb_0026.png
[30/43] Processing rgb_0031.png
[35/43] Processing rgb_0036.png
[40/43] Processing rgb_0041.png
Total merged points before final subsampling: (43146, 3)
Total merged points after subsampling: (43146, 3)
PLY written with 43146 points -> C:\Users\admin\Documents\Vicki\School stuff\SEGP\sample_001\ffb_multi_frame_numpy.ply
